In [1]:
import ee
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import rasterio

In [2]:
from tgbs_rs.config import AOI_PATHS
from tgbs_rs.utils import geojson_to_ee_geometry
from tgbs_rs.baseline import build_baseline_layers
from tgbs_rs.config import PLOTTING_SCALE_DICT
from tgbs_rs.export_rasters import (
    export_named_layer_to_drive,
    export_selected_layers_to_drive,
)
from tgbs_rs.plotting import plot_baseline_panels_from_rasters

In [3]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project="charrell-personal") 

In [4]:
kwale_aoi = geojson_to_ee_geometry(AOI_PATHS["kwale"])

In [5]:
layers = build_baseline_layers(kwale_aoi)

In [6]:
tasks = export_selected_layers_to_drive(
    layers=layers,
    aoi=kwale_aoi,
    layer_names=["dem", "slope", "hillshade"],
    folder="TGBS_Kwale_Baseline",
    scale_dict=PLOTTING_SCALE_DICT,
    crs="EPSG:4326",
)

Starting export for layer: dem
Starting export for layer: slope
Starting export for layer: hillshade


In [7]:
tasks = export_selected_layers_to_drive(
    layers=layers,
    aoi=kwale_aoi,
    layer_names=["land_cover", "soil_carbon", "bii_all"],
    folder="TGBS_Kwale_Baseline",
    scale_dict=PLOTTING_SCALE_DICT,
    crs="EPSG:4326",
)

Starting export for layer: land_cover
Starting export for layer: soil_carbon
Starting export for layer: bii_all


In [8]:
task = export_named_layer_to_drive(
    layers=layers,
    aoi=kwale_aoi,
    layer_name="canopy_height",
    folder="TGBS_Kwale_Baseline",
    scale_dict=PLOTTING_SCALE_DICT,
    crs="EPSG:4326",
)

In [ ]:
fig, axes = plot_baseline_panels_from_rasters("../outputs/raster")
plt.show()

In [10]:
def inspect_raster_nodata(raster_path):
    """
    Inspect nodata, mask, NaNs, zeros, and value range for a raster.
    """
    raster_path = Path(raster_path)

    with rasterio.open(raster_path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata
        mask_flags = src.mask_flag_enums[0]

        nan_count = int(np.isnan(arr).sum())
        zero_count = int((arr == 0).sum())

        valid = arr[~np.isnan(arr)]

        min_val = float(valid.min()) if valid.size else None
        max_val = float(valid.max()) if valid.size else None

    print(f"\n--- {raster_path.name} ---")
    print(f"nodata value: {nodata}")
    print(f"mask flags: {mask_flags}")
    print(f"nan count: {nan_count}")
    print(f"zero count: {zero_count}")
    print(f"min / max (non-NaN): {min_val} / {max_val}")

In [11]:
def inspect_raster_folder(folder_path):
    """
    Inspect all GeoTIFF rasters in a folder.
    """
    folder_path = Path(folder_path)

    for raster_path in sorted(folder_path.glob("*.tif")):
        inspect_raster_nodata(raster_path)

In [ ]:
inspect_raster_folder("../outputs/raster")